In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [2]:
sentences = [
 "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
]
labels = [1]*15 + [0]*15
labels = np.array(labels)

In [3]:
# Set maximum vocabulary size
vocab_size = 2000

# Create tokenizer
# oov_token handles unknown words
tok = Tokenizer(
    num_words=vocab_size,
    oov_token=""
)

# Build vocabulary from sentences
tok.fit_on_texts(sentences)

# Convert text sentences into sequences of numbers
seqs = tok.texts_to_sequences(sentences)

# Find maximum sentence length
maxlen = max(len(s) for s in seqs)

# Pad sequences so all have same length
# 'post' adds padding at the end
X = pad_sequences(
    seqs,
    maxlen=maxlen,
    padding='post'
)

# Target labels
y = labels

In [4]:

X[0]

array([ 3, 26,  2,  7,  0], dtype=int32)

In [5]:
# Size of word embedding vectors
# Each word will be represented using 16 numbers
embed_dim = 16

# Number of neurons/units in RNN layer
# Controls learning capacity of the RNN
rnn_units = 8

In [6]:
# Input layer
# Shape = maximum sentence length
inp = Input(
    shape=(maxlen,),
    dtype="int32",
    name='input'
)

# Embedding layer
# Converts word indices into dense vectors
x = Embedding(
    input_dim=vocab_size,
    output_dim=embed_dim,
    mask_zero=True,
    name='embed'
)(inp)

# Create Simple RNN layer
rnn = SimpleRNN(
    units=rnn_units,
    return_sequences=False,
    return_state=False,
    name='simple_rnn'
)

# Pass embedding output into RNN
x_last = rnn(x)

# Output layer
# Sigmoid activation for binary classification
out = Dense(
    1,
    activation='sigmoid',
    name='out'
)(x_last)

# Create complete model
model = Model(
    inputs=inp,
    outputs=out
)

# Compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 5)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 5, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ (None, 8)         │        200 │ embed[0][0],      │
│ (SimpleRNN)         │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.fit(X,y,epochs= 25,batch_size = 8,verbose = 1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.4333 - loss: 0.7031
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6000 - loss: 0.6846
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7000 - loss: 0.6690
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7667 - loss: 0.6527
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8667 - loss: 0.6382
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9333 - loss: 0.6227
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9333 - loss: 0.6062
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9333 - loss: 0.5882
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9333 - loss: 0.5694
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9333 - loss: 0.5508
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9333 - loss: 0.5306
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9333 - loss: 0.5083 


# Create intermediate model

Subtask:
Build a new Keras Model that takes the same input as the original model but outputs the embedding layer and the hidden states of the SimpleRNN layer.

Reasoning: Define an intermediate Keras model with the same input as the original model and outputs from the embedding and simple RNN layers.

In [8]:
# Create intermediate model
# Used to extract outputs from hidden layers

intermediate_model = Model(
    inputs=model.inputs,

    # Get outputs from:
    # 1. Embedding layer
    # 2. Simple RNN layer
    outputs=[
        model.get_layer('embed').output,
        model.get_layer('simple_rnn').output
    ]
)

In [9]:
# Import SimpleRNN with alias
from tensorflow.keras.layers import SimpleRNN as SRNN

# Create new input layer
seq_inp = Input(
    shape=(maxlen,),
    dtype='int32'
)

# Reuse trained embedding layer from original model
seq_emb = model.get_layer('embed')(seq_inp)

# Create new RNN layer
# return_sequences=True returns hidden state at every timestep
rnn_seq = SRNN(
    units=rnn_units,
    return_sequences=True,
    name='rnn_seq'
)

# Pass embeddings through RNN
# Layer gets built automatically
seq_hidden = rnn_seq(seq_emb)

# Copy trained weights from original RNN layer
try:

    # Get weights from trained SimpleRNN layer
    trained_weights = model.get_layer('simple_rnn').get_weights()

    # Set same weights into new sequence RNN
    rnn_seq.set_weights(trained_weights)

    print("Copied RNN weights into sequence-inspection RNN.")

except Exception as e:

    print("Could not copy weights automatically:", e)

# Create inspection model
# Outputs hidden states for all timesteps
inspect_model = Model(
    inputs=seq_inp,
    outputs=seq_hidden
)

# =========================
# Inspect Hidden States
# =========================

# Select one example sentence
idx = 0

# Shape becomes (1, maxlen)
example_seq = X[idx:idx+1]

# Predict hidden states
hidden_seq = inspect_model.predict(example_seq)

# Display results
print("Sentence:", sentences[idx])

print("Token ids:", example_seq)

print("Hidden states per timestep shape:", hidden_seq.shape)

print("Hidden states (timesteps x units):")

# Round values for better readability
print(np.round(hidden_seq[0], 3))

Copied RNN weights into sequence-inspection RNN.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 357ms/step
Sentence: I love this product
Token ids: [[ 3 26  2  7  0]]
Hidden states per timestep shape: (1, 5, 8)
Hidden states (timesteps x units):
[[ 0.041  0.012 -0.019  0.088  0.024 -0.06   0.017 -0.057]
 [ 0.096 -0.058  0.106 -0.049  0.08  -0.097  0.015 -0.045]
 [-0.     0.096  0.093 -0.045 -0.138  0.113  0.066  0.098]
 [ 0.012 -0.125  0.182  0.213  0.213 -0.174 -0.044 -0.068]
 [ 0.012 -0.125  0.182  0.213  0.213 -0.174 -0.044 -0.068]]
